# 《PythAPCS123》單元 13-2：語法錯誤（SyntaxError）深度排查與 Traceback 閱讀心法

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/johnnyy-lab/APCS1to3/blob/main/PythAPCS123_13-2_syntax_errors_and_traceback_decoding.ipynb)

**適合對象**：程式設計初學者（完全零基礎） / APCS 扎根學習者
**核心目標**：克服初學者面對直譯器紅色英文報錯（Traceback）的內心恐懼，徹底搞懂「錯誤行號（Line Number）」與「游標指標（^）」的真實座標意義，地毯式拆解考場最常見的 7 大高頻語法與詞法雷區：漏冒號、括號不對稱與引號混用、上一行漏關括號引爆下一行報錯、縮排不一致與 TabError、全形空白與中文標點幽靈、賦值與非法運算子（含 C++ 習慣 i++ 誤用）、關鍵字拼錯與保留字衝突、以及 f-string 格式語法地雷。練就 10 秒精準鎖定並修復 SyntaxError 的必備考場除錯神技！


### 13.2.1 直譯器編譯檢查期：為什麼程式碼尚未執行就先報錯？

許多初學者在剛開始學習 Python 時，常常存在一種根深蒂固的直覺誤解：「Python 是直譯語言，所以它一定是一行一行讀取、讀到哪一行就執行哪一行；因此，只要我的程式前面幾行沒寫錯，電腦至少應該先把前幾行的 `print()` 印出來，等跑到出錯的那一行才會當機停下來吧？」

當初學者滿懷期待地按下執行鍵，卻發現畫面上「連半個字都沒印出來」，直譯器就立刻在第 10 行噴出紅色的 `SyntaxError` 時，往往會感到無比困惑。這是因為 Python 在執行任何程式碼之前，內部必須先經歷一段極其迅速的**「語法剖析與位元組編譯期（Parsing & Bytecode Compilation Phase）」**。直譯器會先把整份原始程式碼（.py 檔案）當作一篇完整的文章從頭到尾掃描一遍，並將人類看得懂的程式文字轉換為電腦底層能夠高效處理的「位元組碼（Bytecode）」。

只要整篇文章中有任何一個單字拼錯、括號沒關、縮排錯位或漏掉冒號，語法剖析器（Parser）就會認定整份文法結構不合規格，直譯器會當機立斷「拒絕啟動虛擬機（Python Virtual Machine, PVM）」，連第 1 行的 `print("程式開始")` 都不會執行！理解這個靜態檢查階段至關重要：語法錯誤是直譯器在門口就把你攔截下來，它與程式執行到一半崩潰的「執行時期錯誤（Runtime Error）」有著本質上的不同。


In [ ]:
# 13.2.1 程式碼演示：對比「語法錯誤編譯期攔截」vs「執行時期崩潰」
import sys

# 案例 A: 語法錯誤（SyntaxError）—— 整份程式連第 1 行都不會執行！
syntax_error_code = '''
print("[步驟 1] 程式準備啟動")
print("[步驟 2] 讀取環境變數")
# 第 5 行故意漏掉冒號
if 10 > 5
    print("[步驟 3] 條件成立")
'''

print("=== 案例 A: 執行包含 SyntaxError 的代碼 ===")
try:
    exec(syntax_error_code)
except SyntaxError as e:
    print(f"❌ 攔截到 SyntaxError！錯誤發生於第 {e.lineno} 行: {e.msg}")
    print("觀察重點：注意上方完全沒有印出『步驟 1』或『步驟 2』！編譯期即被攔截。")

# 案例 B: 執行時期錯誤（RuntimeError）—— 前面正常印出，直到出錯行才崩潰！
runtime_error_code = '''
print("[步驟 1] 程式順利啟動")
print("[步驟 2] 正在進行前置計算")
# 語法完全正確，但數值邏輯引爆除以零崩潰
danger_val = 10 / 0
print("[步驟 3] 運算完成")
'''

print("\n=== 案例 B: 執行包含 RuntimeError 的代碼 ===")
try:
    exec(runtime_error_code)
except ZeroDivisionError as e:
    print(f"⚠️ 攔截到 ZeroDivisionError: {e}")
    print("觀察重點：步驟 1 與步驟 2 均已成功印出，直到第 6 行才在執行階段中途暴斃！")

# 案例 C: 多重語法錯誤掩蓋現象 —— 直譯器永遠只會回報「第一個撞到的錯誤」！
multi_error_code = '''
if x > 0       # 錯誤 1: 漏冒號
    print(x)
for i in range(5) # 錯誤 2: 漏冒號
    print(i)
'''
print("\n=== 案例 C: 多重語法錯誤只報第一個 ===")
try:
    compile(multi_error_code, "<multi_test>", "exec")
except SyntaxError as e:
    print(f"直譯器回報行號: 第 {e.lineno} 行（錯誤 1）")
    print("提示：修復了第 2 行的錯誤後，再次編譯才會暴露第 4 行的錯誤 2！")


### 13.2.1 語法重點回顧與核心觀念提煉

在剛才的三重案例驗證中，我們建立了兩大極具價值的除錯心智模型：
1. **編譯期攔截 vs 執行期中斷**：
   - `SyntaxError`（語法錯誤）：發生於編譯期，整份程式連第 1 行都進不去，在 OJ 評判系統直接拿下 `CE（Compile Error）`。
   - `Exception`（執行時期例外）：語法完全合法，程式能正常啟動並執行前段，直到出錯那一行才中途崩潰，OJ 判定為 `RE（Runtime Error）`。
2. **連鎖排查定律**：因為直譯器在遇到第一個語法錯誤時就會立刻停止掃描，所以當你修復了報錯的第 5 行後，再次執行可能又在第 15 行跳出新報錯。這是完全正常的現象，切勿慌張，只需順著 Traceback 由上而下逐一消滅即可！


In [ ]:
# 13.2.1 學生實作練習：程式碼執行階段診斷器
# 任務說明：實作 diagnose_code_phase(code_str) 函式
# 判斷傳入的 code_str 屬於哪種階段狀態：
# 1. 若語法檢查失敗（引發 SyntaxError），回傳 "COMPILE_ERROR"
# 2. 若能順利通過 compile() 編譯，回傳 "SYNTAX_OK"

def diagnose_code_phase(code_str: str) -> str:
    # 請在此處使用 compile 搭配 try-except 進行階段判定
    try:
        compile(code_str, filename="<test>", mode="exec")
        return "SYNTAX_OK"
    except SyntaxError:
        return "COMPILE_ERROR"

# 測試用例
code_valid = "a = 10\nb = 20\nprint(a + b)"
code_invalid = "for i in range(10)\n    print(i)"
print("測試 1 (合法代碼):", diagnose_code_phase(code_valid))
print("測試 2 (漏冒號代碼):", diagnose_code_phase(code_invalid))


In [ ]:
# 13.2.1 單元測試驗證
assert diagnose_code_phase("x = 5\nprint(x)") == "SYNTAX_OK"
assert diagnose_code_phase("if True\n    pass") == "COMPILE_ERROR"
assert diagnose_code_phase("arr = [1, 2, 3") == "COMPILE_ERROR"
assert diagnose_code_phase("def foo(x):\n    return x * 2") == "SYNTAX_OK"
assert diagnose_code_phase("print('hello' + 5)") == "SYNTAX_OK", "語法合法，型態錯誤屬於執行期而非語法期"
print("🎉 13.2.1 所有測試通過！成功掌握編譯期與執行期核心機制！")


### 13.2.2 看懂 Traceback 座標軸：錯誤行號與游標指標 `^`（上一行漏括號嫁禍陷阱）

每當終端機噴出一整片紅色的英文錯誤追蹤訊息（Traceback）時，許多零基礎學生的第一反應往往是「眼神迴避、心跳加速、不知所措」，甚至下意識立刻把整個視窗關掉重開。請記住：**Traceback 不是電腦在罵你，而是電腦在向你發出最精準的求救訊號！**

在所有語法錯誤的 Traceback 訊息中，直譯器提供了一套非常精準的「二維空間座標定位系統」：
1. **第一維度：檔案名稱與行號（`File "...", line X`）**：這是直譯器的 Y 軸座標。它明確告訴你，解析器是在第幾行撞牆的。
2. **第二維度：原始程式碼片段（Code Text）**：直譯器會把出問題的那一行程式碼完整印出來給你看。
3. **第三維度：游標箭頭指標（Caret Symbol `^` 或波浪線 `~~~`）**：這是直譯器的 X 軸座標。在 Python 3.10+ 版本中，強化版直譯器會用一個或多個 `^` 與 `~`，極其精準地指著它「看到哪一個字元發現不合法」。
4. **第四維度：錯誤類型與原因描述（Error Message）**：在最底下一行，如 `SyntaxError: expected ':'`（預期需要冒號）或 `SyntaxError: unmatched ')'`（不匹配的多餘右括號）。

⚠️ **考場最大盲區：上一行漏括號，下一行背黑鍋！**
如果上一行的小括號、中括號或引號沒有關閉（例如 `total = (10 + 20`），Python 直譯器會認為該表達式尚未結束，繼續往下讀取第二行；當它讀到第二行的第一個指令時，才會發現文法完全接不起來，於是在第二行噴出 `SyntaxError`！這時**游標指著第二行，但真正的罪魁禍首 100% 在上一行**！


In [ ]:
# 13.2.2 程式碼演示：解析 Traceback 座標軸與「上一行未閉合」的嫁禍陷阱
import traceback

def analyze_syntax_traceback(code_text, label):
    print(f"=== 分析案例 [{label}] ===")
    lines = code_text.strip().split('\n')
    for idx, l in enumerate(lines, 1):
        print(f"  行 {idx:02d}: {l}")
    try:
        compile(code_text, "<test_file>", "exec")
        print("  --> [PASS] 語法檢查通過！\n")
    except SyntaxError as e:
        print(f"  --> [直譯器報錯座標] 行號: 第 {e.lineno} 行, 欄位: 第 {e.offset} 字元")
        print(f"  --> [出錯代碼片段] {repr(e.text.strip() if e.text else '')}")
        indent = " " * (e.offset - 1) if e.offset else ""
        print(f"  --> [游標指標箭頭] {indent}^")
        print(f"  --> [診斷訊息描述] {e.msg}")
        if e.lineno > 1 and "invalid syntax" in e.msg:
            print("  💡 [考場避坑提示] 報錯在該行開頭？請務必抬頭檢查『前一行』是否括號或引號未閉合！")
        print()

# 案例 1: 當前行明確出錯（多餘運算子）
analyze_syntax_traceback("a = 10\nb = 20 + * 5\nprint(a + b)", "當前行明確錯誤")

# 案例 2: 經典黑鍋陷阱 —— 第 1 行漏右括號，報錯卻在第 2 行！
analyze_syntax_traceback("total = sum([1, 2, 3\nans = total * 2\nprint(ans)", "上一行未閉合嫁禍陷阱")

# 案例 3: 巢狀呼叫末尾漏括號
analyze_syntax_traceback("val = int(input().split()[0]\nprint(val)", "雙層巢狀呼叫漏括號")


### 13.2.2 語法重點回顧與核心觀念提煉

閱讀 Traceback 的標準五部曲：
1. **看最後一行**：先讀最底部的錯誤描述（如 `SyntaxError: expected ':'`、`SyntaxError: unterminated string literal`），搞懂電腦到底想要什麼。
2. **看行號與箭頭**：找到 `File "...", line X` 與 `^` 箭頭，確認電腦撞牆的精確字元。
3. **啟動「抬頭望上一行」雷達**：如果箭頭指在某個變數的開頭、或是看似毫無問題的指令上，**立刻抬頭看前一行**！90% 的機率是前一行少打了一個右括號 `)`、右中括號 `]` 或引號。
4. **利用現代編輯器的彩虹括號對齊**：在 Colab 或 VSCode 中，將游標點在括號旁邊，觀察對應的配對括號是否有亮起，能在 3 秒內揪出失蹤的括號！


In [ ]:
# 13.2.2 學生實作練習：括號配對平衡檢查器
# 任務說明：實作 is_brackets_balanced(expr) 函式
# 檢查字串 expr 中的小括號 ()、中括號 []、大括號 {} 是否完全對稱且正確閉合
# 若完全合法回傳 True；若有未閉合或類型錯配回傳 False

def is_brackets_balanced(expr: str) -> bool:
    stack = []
    pairs = {')': '(', ']': '[', '}': '{'}
    # 請在此處使用堆疊實作括號平衡檢驗
    for ch in expr:
        if ch in "([{":
            stack.append(ch)
        elif ch in ")]}":
            if not stack or stack[-1] != pairs[ch]:
                return False
            stack.pop()
    return len(stack) == 0

# 測試用例
print("測試合法:", is_brackets_balanced("[(1 + 2) * 3]"))
print("測試未閉合:", is_brackets_balanced("total = sum([1, 2, 3"))
print("測試錯配:", is_brackets_balanced("[1, 2, 3)"))


In [ ]:
# 13.2.2 單元測試驗證
assert is_brackets_balanced("(1 + 2)") == True
assert is_brackets_balanced("[(a + b) * {c - d}]") == True
assert is_brackets_balanced("((())") == False
assert is_brackets_balanced("[1, 2, 3)") == False
assert is_brackets_balanced(")") == False
assert is_brackets_balanced("") == True
print("🎉 13.2.2 所有測試通過！成功破解括號不對稱與座標定位盲點！")


### 13.2.3 考場語法雷區一：結構控制漏冒號 `:` 深度排查

在 APCS 考場的高壓環境下，語法錯誤統計數據顯示：**「漏打冒號 `:`」**長年穩坐扣分排行榜的第一名寶座！
在 Python 的文法設計與抽象語法樹（AST）規範中，冒號具有無可取代的神聖地位：它是直譯器判定「複合語句標頭（Statement Header）宣告完畢、準備引導進入下一個縮排程式碼區塊（Suite）」的唯一合法符號與語法通行證。如果遺漏了冒號，直譯器根本無法界定條件判斷何時結束、區塊程式碼何時開始，只能直接中斷並丟出 `SyntaxError: expected ':'`。

考場中最容易因為手指打字太快而遺漏冒號的六大關鍵語法族群：
1. **條件分支族**：`if condition:`、`elif condition:`、`else:`。特別注意：初學者常把心力放在複雜的條件運算式上，寫完條件就急著按換行鍵，導致 `if` 漏冒號；或者是寫到分支最後的 `else` 時，因為後面不需要再寫任何條件式，大腦放鬆之下極高頻率直接按下 Enter，造成考場最高頻手滑！
2. **迴圈控制族**：`for item in sequence:`、`while condition:`。在寫多重巢狀迴圈（例如雙層 for 迴圈走訪二維矩陣）時，內層迴圈常因匆促而漏掉末端冒號。
3. **函式定義族**：`def function_name(params):`。在撰寫遞迴函式或自訂輔助演算法時，宣告參數清單括號關閉後，務必補上冒號。
4. **例外防禦族**：`try:`、`except Exception:`、`finally:`。與 `else` 類似，`try:` 與 `finally:` 自身單獨成行，初學者極易遺漏。

⚠️ **C/C++ 與 Java 考生的跨語言思維陷阱**：
許多從 C/C++ 或 Java 跨考 Python 的學生，習慣使用小括號包覆條件式並以大括號定義區塊，例如下意識寫出 `if (x > 0) {` 或 `while (x > 0)`，甚至在行尾加上分號 `;`。在 Python 中，大括號是用來表示集合（set）或字典（dict）的字面常數，出現在條件式尾端會立刻觸發直譯器嚴重的語法崩潰！牢記冒號的引導規則，並養成每寫完複合標頭立刻打冒號的肌肉記憶，是徹底杜絕編譯失敗（Compile Error）的第一防線。


In [ ]:
# 13.2.3 程式碼演示：漏冒號與 C 語言習慣四大雷區
print("--- 雷區 1: if / elif / else 各環節漏冒號示範 ---")
def demonstrate_colon_pitfall():
    test_cases = [
        ("if x > 10\n    pass", "if 漏冒號"),
        ("elif x == 5\n    pass", "elif 漏冒號"),
        ("else\n    pass", "else 漏冒號 (考場最高頻手滑！)"),
        ("for i in range(5)\n    pass", "for 迴圈漏冒號"),
        ("def calc(a, b)\n    return a + b", "def 函式宣告漏冒號")
    ]
    for code, desc in test_cases:
        try:
            compile(code, "<colon_test>", "exec")
        except SyntaxError as e:
            print(f"  [攔截成功] {desc} -> 報錯行 {e.lineno}: {e.msg}")

demonstrate_colon_pitfall()

print("\n--- 雷區 2: C/C++ 習慣語法在 Python 中引爆語法錯誤 ---")
c_style_code = '''
if (x > 10) {
    print("x is big")
}
'''
try:
    compile(c_style_code, "<c_style>", "exec")
except SyntaxError as e:
    print(f"  [C語言大括號大忌] 報錯: {e.msg} (Python 用縮排而非大括號界定區塊！)")


### 13.2.3 語法重點回顧與核心觀念提煉

冒號防身守則：
1. **只要新開區塊，必定以冒號收尾**：打完 `if`、`elif`、`else`、`for`、`while`、`def` 的當下，右手小指立刻按下鍵盤上的 `:` 鍵，養成肌肉記憶。
2. **else 與 finally 必帶冒號**：這兩個關鍵字後方不接任何條件式，初學者最常因此直接換行而遺漏冒號。
3. **拒絕 C 風格大括號**：在 Python 條件語句中，絕不要在行尾寫 `{`！


In [ ]:
# 13.2.3 學生實作練習：自動補齊語法結構冒號
# 任務說明：實作 auto_fix_colons(line) 函式
# 給定單行程式碼字串 line（例如 "if x > 5" 或 "    else"）
# 若該行屬於需要冒號結尾的複合語句開頭（以 if, elif, else, for, while, def 開頭）
# 且結尾尚未包含冒號 ':'，請自動在末尾補上 ':'；其餘情況保持原樣回傳！

def auto_fix_colons(line: str) -> str:
    stripped = line.rstrip()
    leading_spaces = line[:len(line) - len(line.lstrip())]
    pure_text = stripped.lstrip()
    
    keywords = ("if ", "elif ", "else", "for ", "while ", "def ")
    needs_colon = any(pure_text.startswith(kw) or pure_text == kw.strip() for kw in keywords)
    
    if needs_colon and not pure_text.endswith(":"):
        return stripped + ":"
    return line

# 測試用例
print("修復 if:", auto_fix_colons("if x > 10"))
print("修復 else (帶縮排):", repr(auto_fix_colons("    else")))
print("已有冒號:", auto_fix_colons("for i in range(5):"))


In [ ]:
# 13.2.3 單元測試驗證
assert auto_fix_colons("if x == 1") == "if x == 1:"
assert auto_fix_colons("    else") == "    else:"
assert auto_fix_colons("def foo(x):") == "def foo(x):"
assert auto_fix_colons("x = 10") == "x = 10"
assert auto_fix_colons("while True") == "while True:"
print("🎉 13.2.3 所有測試通過！成功建立冒號肌肉記憶！")


### 13.2.4 考場語法雷區二：括號不對稱與引號混用未閉合

在 Python 程式語言的文法解析體系中，括號與引號扮演著「界定邊界（Delimiters）」的核心角色。它們在語法規則中必須如同鏡像般成雙成對、精準閉合。一旦出現「左側開了門、右側卻忘記關閉」或是「前後括號類型錯配」，直譯器的語法剖析器（Parser）就會瞬間迷失方向，導致整個編譯流程直接癱瘓中斷。

在解題與考場高壓下，最常見的四大括號與引號翻車情境：
1. **括號類型交叉錯配（Mismatched Brackets）**：
   以小括號開頭，卻以中括號結束，例如 `nums = (1, 2, 3]`。Python 直譯器在 3.10+ 版本後能精準辨識並回報 `SyntaxError: closing parenthesis ']' does not match opening parenthesis '('`，清楚指出兩端括號的不相容性。
2. **引號前後混用未對齊（Mismatched Quotes）**：
   例如 `'hello"`（前單後雙）或 `"world'`（前雙後單）。直譯器看到單引號開頭時，會持續尋找下一個單引號作為終止標記；若中途遇到雙引號，直譯器只會把它當成一般字元繼續往後吞噬，直到整行結束都找不到匹配的單引號，最終噴出 `SyntaxError: unterminated string literal`。
3. **字串內部撇號（Apostrophe）提前截斷字串**：
   當字串外層使用單引號包裹，而字串內部包含英文縮寫或撇號時（例如 `msg = 'It's fine'`），第 2 個單引號會被直譯器誤認為字串的終點（字串變成 `'It'`），而後面緊跟著的 `s fine'` 瞬間變成沒有變數宣告、毫無文法邏輯的非法字元碎片，引爆致命的語法崩潰！解決方案是外層改用雙引號 `"It's fine"`，或是使用反斜線進行跳脫逃逸 `'It\'s fine'`。
4. **多行文件字串（Docstrings）未正常收尾**：
   在撰寫多行說明文字或長字串時，開頭使用了三個單引號或三個雙引號，但在結尾處少打了一個引號（變成只有兩個引號），直譯器便會將後續所有正常的 Python 程式碼全部當作字串內容吞沒，直到檔案結尾拋出 `unterminated triple-quoted string literal`！


In [ ]:
# 13.2.4 程式碼演示：括號錯配、引號混用與字串內部單引號陷阱
print("--- 案例 1: 括號類型錯配（圓括號配方括號）---")
mismatched_brackets_code = "nums = (1, 2, 3]"
try:
    compile(mismatched_brackets_code, "<bracket_test>", "exec")
except SyntaxError as e:
    print(f"  [捕捉錯誤 1] SyntaxError: {e.msg} (行 {e.lineno})")

print("\n--- 案例 2: 前單後雙引號混用 ---")
mixed_quotes_code = 'text = "hello\''
try:
    compile(mixed_quotes_code, "<quote_test>", "exec")
except SyntaxError as e:
    print(f"  [捕捉錯誤 2] SyntaxError: {e.msg}")

print("\n--- 案例 3: 字串內部縮寫撇號提早閉合 ---")
# 示範字串內部的撇號問題
inner_quote_code = "greeting = 'It's a beautiful day'"
try:
    compile(inner_quote_code, "<inner_test>", "exec")
except SyntaxError as e:
    print(f"  [捕捉錯誤 3] SyntaxError: {e.msg}")
    print("  💡 正解：含有撇號的句子外層必用雙引號：\"It's a beautiful day\"！")


### 13.2.4 語法重點回顧與核心觀念提煉

括號與引號閉合三大紀律：
1. **同種對稱原則**：`(` 對 `)`、`[` 對 `]`、`{` 對 `}`，絕不 क्रॉस 混用。
2. **單雙引號互補原則**：文字內含單引號（如 `it's`），外層就用雙引號 `"..."`；文字內含雙引號（如對話引言），外層就用單引號 `'...'`。
3. **跨行大字串用三引號**：`'''...'''` 能安全容納所有換行與各類單雙引號。


In [ ]:
# 13.2.4 學生實作練習：安全字串引號包覆器
# 任務說明：實作 safe_quote_string(raw_text) 函式
# 給定任意原始文字 raw_text（可能包含單引號 ' 或雙引號 "）
# 產生一個合法的 Python 字串宣告表達式：
# 1. 若 raw_text 包含單引號但不含雙引號，外層以雙引號包覆：f'"{raw_text}"'
# 2. 若 raw_text 包含雙引號但不含單引號，外層以單引號包覆：f"'{raw_text}'"
# 3. 若同時包含單雙引號，使用三引號包覆：f"'''{raw_text}'''"
# 回傳該表達式，並確保其能順利被 eval() 解析！

def safe_quote_string(raw_text: str) -> str:
    has_single = "'" in raw_text
    has_double = '"' in raw_text
    
    if has_single and not has_double:
        return f'"{raw_text}"'
    elif has_double and not has_single:
        return f"'{raw_text}'"
    elif has_single and has_double:
        return f"'''{raw_text}'''"
    else:
        return f'"{raw_text}"'

# 測試用例
print("包含單引號:", safe_quote_string("It's Python"))
print("包含雙引號:", safe_quote_string('He said "Hello"'))
print("同時包含:", safe_quote_string('He said "It\'s cool"'))


In [ ]:
# 13.2.4 單元測試驗證
t1 = safe_quote_string("It's sunny")
assert eval(t1) == "It's sunny"

t2 = safe_quote_string('Say "Hi"')
assert eval(t2) == 'Say "Hi"'

t3 = safe_quote_string('He said "It\'s fine"')
assert eval(t3) == 'He said "It\'s fine"'
print("🎉 13.2.4 所有測試通過！成功建立括號與引號互補防衛體系！")


### 13.2.5 考場語法雷區三：縮排不一致 IndentationError 與 TabError 混用災難

在 Python 中，「縮排（Indentation）」不是單純為了排版美觀，而是程式邏輯區塊的唯一標誌。這與 C/C++、Java 使用大括號 `{}` 的機制截然不同。

考場最常引爆縮排災難的三大情境：
1. **`IndentationError: unexpected indent`（意外的多餘縮排）**：在不該縮排的地方多敲了空格，例如在最外層的主程式第 1 行前方手滑按了一個空白鍵。
2. **`IndentationError: expected an indented block`（預期需要縮排區塊）**：在 `if` 或 `def` 冒號下方，留下了空行而沒有寫任何指令。若想暫時留白，必須明確寫下 `pass` 佔位！
3. **`TabError: inconsistent use of tabs and spaces in indentation`（Tab 與空格混用大忌）**：
   在同一個程式碼區塊中，某一行用了 4 個空格，另一行卻按了 Tab 鍵。雖然在某些文字編輯器中它們看起來寬度一樣，但直譯器會認定縮排字元不一致而引爆 `TabError`！

養成良好縮排習慣：考場中嚴格統一「全數按 4 個半形空格」或「全數按 Tab」！


In [ ]:
# 13.2.5 程式碼演示：意外縮排、空區塊未寫 pass 與 Tab 混用錯誤
print("--- 縮排錯誤 1: 行首意外縮排 unexpected indent ---")
code_indent_1 = " a = 10\nprint(a)" # 第 1 行前方多了 1 個空格
try:
    compile(code_indent_1, "<indent_test>", "exec")
except IndentationError as e:
    print(f"  [捕捉成功 1] IndentationError: {e.msg} (第 {e.lineno} 行)")

print("\n--- 縮排錯誤 2: 冒號後留空 expected an indented block ---")
code_indent_2 = "if True:\n# 只有註解\nprint('done')"
try:
    compile(code_indent_2, "<indent_test>", "exec")
except IndentationError as e:
    print(f"  [捕捉成功 2] IndentationError: {e.msg} (第 {e.lineno} 行)")
    print("  💡 正解：若暫時不寫執行語句，必須在冒號下方填寫 pass ！")

print("\n--- 縮排錯誤 3: 空格與 Tab 混用 TabError ---")
# 模擬一行用空格、下一行用 Tab 的混用代碼
code_tab_mix = "def foo():\n    x = 1\n\ty = 2\n    return x + y"
try:
    compile(code_tab_mix, "<tab_test>", "exec")
except TabError as e:
    print(f"  [捕捉成功 3] TabError: {e.msg}")


### 13.2.5 語法重點回顧與核心觀念提煉

縮排防禦兩大鐵則：
1. **縮排嚴格統一**：在編輯器設定中啟用「將 Tab 鍵自動轉換為 4 個半形空格」（Insert Spaces），從物理層面杜絕 TabError。
2. **空區塊必填 `pass`**：撰寫程式骨架（Skeleton）時，只要打了冒號，立刻在縮排下一行寫下 `pass`，確保整份代碼能隨時順利編譯！


In [ ]:
# 13.2.5 學生實作練習：自動填充空區塊 pass 修正器
# 任務說明：實作 fill_empty_blocks_with_pass(code_str) 函式
# 檢查一段程式碼：若某一行的結尾是冒號 ':'，且其下一行不是縮排區塊（縮排長度小於等於該行）
# 自動在其下方插入一行縮排加上 "pass" 的語句，使代碼能合法通過編譯！

def fill_empty_blocks_with_pass(code_str: str) -> str:
    lines = code_str.split('\n')
    new_lines = []
    
    for i in range(len(lines)):
        new_lines.append(lines[i])
        stripped = lines[i].rstrip()
        if stripped.endswith(":"):
            # 計算當前行的縮排
            cur_indent = len(lines[i]) - len(lines[i].lstrip())
            # 檢查下一行是否為縮排區塊
            if i + 1 >= len(lines) or (len(lines[i + 1]) - len(lines[i + 1].lstrip())) <= cur_indent:
                # 補上一行帶有額外 4 空格的 pass
                new_lines.append(" " * (cur_indent + 4) + "pass")
                
    return '\n'.join(new_lines)

# 測試用例
empty_block_code = "if True:\nprint('finished')"
fixed = fill_empty_blocks_with_pass(empty_block_code)
print("修正後代碼:\n" + fixed)


In [ ]:
# 13.2.5 單元測試驗證
raw = "if x > 0:\nprint(x)"
res = fill_empty_blocks_with_pass(raw)
assert "    pass" in res
# 驗證修正後能夠通過編譯
compile(res, "<test>", "exec")
print("🎉 13.2.5 所有測試通過！成功駕馭縮排與 pass 佔位心法！")


### 13.2.6 語法結構雷區四：隱形字元災難——全形空白 `\u3000` 與中文標點符號

這是所有零基礎初學者最常遭遇的「靈異事件」！當考生從題目的 PDF 說明、教學講義或是 LINE/網頁複製程式碼範例時，文字中經常夾帶了中文全形空白鍵（Unicode 編碼 `\u3000`，ASCII 值為 12288）。

在螢幕上看，它跟一般英文半形空格長得一模一樣；但 Python 直譯器只認得半形 ASCII 空格（`\x20`，ASCII 值為 32）。一旦遇到全形空白，直譯器會當場引爆令人崩潰的：
```
SyntaxError: invalid character '　' (U+3000)
```

同樣的災難也發生在**「中文全形標點符號」**：
- 中文冒號 `：`（U+FF1A）$\to$ 直譯器無法辨識為結構開關。
- 中文逗號 `，`（U+FF0C）$\to$ 無法切分多變數或引數。
- 中文括號 `（）`（U+FF08/U+FF09）$\to$ 無法當作函式呼叫或運算優先級。
- 中文引號 `“”`（U+201C/U+201D）$\to$ 無法界定字串。

只要具備 Unicode 辨識意識，這類「靈異報錯」就能在 5 秒內現形並徹底淨化！


In [ ]:
# 13.2.6 程式碼演示：全形空白與中文標點符號顯形術
dirty_snippet = "if　x > 5：\n　　print（x，10）"

print("=== 原始代碼（肉眼看似完全正常）===")
print(dirty_snippet)

print("\n=== 底層 Unicode 碼位透視鏡 ===")
for line_idx, line in enumerate(dirty_snippet.split('\n'), 1):
    for ch in line:
        code_point = ord(ch)
        if code_point > 127: # 非標準 ASCII 字符
            print(f"  [行 {line_idx}] 抓到中文字元: '{ch}' -> Unicode U+{code_point:04X} (非半形 ASCII！)")

print("\n=== 嘗試直接編譯（直譯器報警現場）===")
try:
    compile(dirty_snippet, "<fullwidth_test>", "exec")
except SyntaxError as e:
    print(f"  ❌ SyntaxError: {e.msg} (第 {e.lineno} 行)")
    print("  說明：Python 直譯器對全形字元毫不妥協！")


### 13.2.6 語法重點回顧與核心觀念提煉

全形字元終結心法：
1. **考場輸入法鎖定**：進考場打開編輯器後，**第一件事將輸入法切換為純英文模式（ENG / 半形）**！
2. **一鍵替換字典**：
   - 全形空格 `\u3000` $\to$ 半形空格 `' '`
   - 全形冒號 `：` $\to$ 半形冒號 `':'`
   - 全形逗號 `，` $\to$ 半形逗號 `','`
   - 全形括號 `（）` $\to$ 半形括號 `'()'`


In [ ]:
# 13.2.6 學生實作練習：全方位全形字元淨化器
# 任務說明：實作 deep_sanitize_code(code_str) 函式
# 深度替換字串中所有的非 ASCII 標點與空白符號：
# 1. '\u3000' -> ' '
# 2. '：' -> ':'
# 3. '，' -> ','
# 4. '（' -> '('，'）' -> ')'
# 5. '“' -> '"'，'”' -> '"'
# 回傳完全合規的半形程式碼，確保能通過 compile()！

def deep_sanitize_code(code_str: str) -> str:
    mapping = {
        '\u3000': ' ',
        '：': ':',
        '，': ',',
        '（': '(',
        '）': ')',
        '“': '"',
        '”': '"'
    }
    result = code_str
    for k, v in mapping.items():
        result = result.replace(k, v)
    return result

# 測試用例
sample_dirty = "for　i　in range（5）：\n　　print（“Case”，i）"
cleaned = deep_sanitize_code(sample_dirty)
print("淨化後代碼:\n" + cleaned)


In [ ]:
# 13.2.6 單元測試驗證
raw = "if　x == 10：\n    print（“PASS”，x）"
res = deep_sanitize_code(raw)
assert '\u3000' not in res
assert '：' not in res
assert '，' not in res
assert '（' not in res
assert '“' not in res
compile(res, "<test>", "exec")
print("🎉 13.2.6 所有測試通過！徹底粉碎全形空白與中文標點隱形幽靈！")


### 13.2.7 語法結構雷區五：運算子與賦值誤用（單等號、非法賦值目標與 i++ 誤區）

在變數運算與條件判斷中，運算子的位置與符號形態常因初學者的舊習慣而引發致命的 SyntaxError。

高頻三大運算子語法雷區：
1. **條件式誤用單等號賦值**：
   在 `if` 或 `while` 中寫下 `if x = 5:`。Python 語法規定條件表達式內部不得進行常規賦值，直譯器會明確警告：`SyntaxError: invalid syntax. Maybe you meant '==' or ':='?`。
2. **非法賦值目標（Cannot assign to literal / expression）**：
   例如寫出 `10 = x`（將變數賦給常數），或 `a + b = 10`（將數值賦給運算式）。等號左側「永遠必須是單一變數箱子」！
3. **C/C++ 自增運算子 `i++` / `i--` 誤用**：
   在 C/C++/Java 中常見的 `i++`，**在 Python 中是完全不存在的語法**！
   - 寫下 `i++` 會當場引發 `SyntaxError: invalid syntax`！
   - *注意*：若寫成前綴 `++i`，Python 不會報錯，但它不會加一，而是解讀為正正得正 `+(+i)`，造成嚴重的邏輯臭蟲！
   - **唯一正解**：永遠寫 **`i += 1`**！
4. **連續非法運算子**：例如手滑打出 `x = 10 + * 5`。


In [ ]:
# 13.2.7 程式碼演示：運算子四大翻車現場剖析
print("--- 案例 1: 條件式中誤用單等號賦值 ---")
try:
    compile("if score = 100:\n    pass", "<eq_test>", "exec")
except SyntaxError as e:
    print(f"  [捕捉錯誤 1] {e.msg} (提示 Maybe you meant '=='?)")

print("\n--- 案例 2: 非法賦值目標（賦值給常數或表達式）---")
try:
    compile("100 = total", "<assign_literal>", "exec")
except SyntaxError as e:
    print(f"  [捕捉錯誤 2] {e.msg}")

try:
    compile("x + y = 20", "<assign_expr>", "exec")
except SyntaxError as e:
    print(f"  [捕捉錯誤 2b] {e.msg}")

print("\n--- 案例 3: C 語言習慣 i++ 誤用 ---")
try:
    compile("counter++", "<inc_test>", "exec")
except SyntaxError as e:
    print(f"  [捕捉錯誤 3] {e.msg} (Python 嚴禁 i++！必須寫 counter += 1)")


### 13.2.7 語法重點回顧與核心觀念提煉

運算子安全自律守則：
1. **問問題用雙等號 `==`，存資料用單等號 `=`**。
2. **等號左側只能是變數名稱**（`x = ...`），不可是常數或算式。
3. **徹底戒除 `++` 習慣**：累加永遠使用複合賦值運算子 `+= 1`，簡潔且符合 Pythonic 規範！


In [ ]:
# 13.2.7 學生實作練習：C 風格自增運算子轉譯器
# 任務說明：實作 convert_c_increment_to_python(line) 函式
# 檢查單行程式碼字串 line：
# 1. 若該行包含 "var++"（例如 "i++" 或 "count++"），將其改寫為標準的 Python "var += 1"
# 2. 若該行包含 "var--"（例如 "i--"），將其改寫為標準的 Python "var -= 1"
# 3. 若無此語法保持原樣回傳！

def convert_c_increment_to_python(line: str) -> str:
    stripped = line.strip()
    if stripped.endswith("++"):
        var_name = stripped[:-2].strip()
        indent = line[:len(line) - len(line.lstrip())]
        return f"{indent}{var_name} += 1"
    elif stripped.endswith("--"):
        var_name = stripped[:-2].strip()
        indent = line[:len(line) - len(line.lstrip())]
        return f"{indent}{var_name} -= 1"
    return line

# 測試用例
print("轉譯 i++:", convert_c_increment_to_python("i++"))
print("轉譯 count-- (帶縮排):", convert_c_increment_to_python("    count--"))
print("正常 Python 代碼:", convert_c_increment_to_python("x += 5"))


In [ ]:
# 13.2.7 單元測試驗證
assert convert_c_increment_to_python("i++") == "i += 1"
assert convert_c_increment_to_python("    idx--") == "    idx -= 1"
assert convert_c_increment_to_python("a = 10") == "a = 10"
compile(convert_c_increment_to_python("i++"), "<test>", "exec")
print("🎉 13.2.7 所有測試通過！成功建立標準運算子防禦意識！")


### 13.2.8 語法結構雷區六：關鍵字拼寫錯誤、保留字撞車與 f-string 語法地雷

除了標點與符號，文字本身的合法性也是語法期的重要戰場。

高頻三大詞法與格式地雷：
1. **關鍵字手滑拼錯（Misspelled Keywords）**：
   - 考場緊張時常打出 `whlie`（while）、`flase`（False）、`ture`（True）、`eif`（elif）。
   - 若拼錯控制關鍵字，直譯器會拋出 `SyntaxError`；若把布林值拼錯，則會引爆 `NameError: name 'ture' is not defined`！
2. **保留字直接拿來當變數名稱（Using Keywords as Identifiers）**：
   - 試圖宣告變數為 Python 保留字：`class = "甲班"`、`def = 10`、`return = 5`。
   - 因為這些單字是直譯器的最高指令，出現在等號左側時直譯器會認定文法嚴重違規，噴出 `SyntaxError: invalid syntax`！
3. **f-string 格式化字串內部語法錯誤**：
   - 大括號內部表達式未寫完：`f"數值是: {x + }"` $\to$ `SyntaxError: f-string: invalid syntax`。
   - 大括號內部留空：`f"結果: {}"` $\to$ `SyntaxError: f-string: empty expression not allowed`。
   - 大括號內部引號衝突：在舊版 Python 中，雙引號 f-string 內部的大括號若又用了雙引號會引爆解析衝突。


In [ ]:
# 13.2.8 程式碼演示：保留字撞車與 f-string 內部表達式錯誤剖析
import keyword

print("--- 案例 1: 試圖使用 Python 保留字作為變數名稱 ---")
print(f"Python 核心保留關鍵字數量: {len(keyword.kwlist)} 個")
print("部分關鍵字範例:", keyword.kwlist[:10])

bad_var_code = "class = '三年一班'"
try:
    compile(bad_var_code, "<kw_test>", "exec")
except SyntaxError as e:
    print(f"  [捕捉錯誤 1] SyntaxError: {e.msg} (保留字 class 絕不可當變數名！)")

print("\n--- 案例 2: f-string 內部殘留空括號或不完整表達式 ---")
bad_fstring_1 = 'msg = f"數值為: {}"' # 空表達式
try:
    compile(bad_fstring_1, "<fstr_test1>", "exec")
except SyntaxError as e:
    print(f"  [捕捉錯誤 2a] f-string 空大括號: {e.msg}")

bad_fstring_2 = 'msg = f"結果為: {10 + }"' # 算式沒寫完
try:
    compile(bad_fstring_2, "<fstr_test2>", "exec")
except SyntaxError as e:
    print(f"  [捕捉錯誤 2b] f-string 算式未完: {e.msg}")


### 13.2.8 語法重點回顧與核心觀念提煉

變數命名與 f-string 安全準則：
1. **變數命名安全檢驗**：使用 `keyword.iskeyword(name)` 驗證變數名稱是否撞車。若想表達班級，改用 `class_name` 或 `cls`，絕不用孤立的 `class`！
2. **f-string 健全性原則**：大括號 `{}` 內部必須是完整的、單獨拿出來也能順利執行的合法表達式。
3. **純大括號字元轉義**：在 f-string 中如果真的想要印出 `{}` 符號本身，請連續打兩次：`f"{{保留大括號}}"`！


In [ ]:
# 13.2.8 學生實作練習：安全識別字驗證器
# 任務說明：實作 is_safe_identifier(name) 函式
# 檢查傳入的字串 name 是否能安全作為 Python 的變數名稱：
# 1. 必須符合 Python 識別字規則（使用 name.isidentifier() 檢查）
# 2. 絕對不能是 Python 的保留關鍵字（使用 keyword.iskeyword(name) 檢查）
# 兩者皆滿足回傳 True，否則回傳 False！

import keyword

def is_safe_identifier(name: str) -> bool:
    # 請在此處進行雙重檢驗
    if not name.isidentifier():
        return False
    if keyword.iskeyword(name):
        return False
    return True

# 測試用例
print("'my_score' 是否安全:", is_safe_identifier("my_score"))
print("'class' 是否安全:", is_safe_identifier("class")) # 保留字，False
print("'2nd_player' 是否安全:", is_safe_identifier("2nd_player")) # 數字開頭，False


In [ ]:
# 13.2.8 單元測試驗證
assert is_safe_identifier("score") == True
assert is_safe_identifier("total_sum") == True
assert is_safe_identifier("_temp") == True
assert is_safe_identifier("class") == False
assert is_safe_identifier("def") == False
assert is_safe_identifier("for") == False
assert is_safe_identifier("123abc") == False
print("🎉 13.2.8 所有測試通過！徹底避開關鍵字撞車與詞法地雷！")


### 13.2.9 考場 1 分鐘語法除錯實戰 SOP：多重語法地雷綜合排查與極速修正演練

在前面的 8 個子單元中，我們將直譯器編譯檢查期、Traceback 座標軸、冒號、括號引號、縮排、全形字元、運算子賦值以及關鍵字保留字等所有語法痛點進行了地毯式的剖析。

當你坐在 APCS 考場的高壓環境中，面對一段充滿多重語法硬傷的代碼時，請啟動這套**「考場 1 分鐘極速語法除錯 SOP」**：

```mermaid
flowchart TD
    A["按下執行 / 送出評判"] --> B{"直譯器噴出紅字 SyntaxError?"}
    B -- "否 (順利進入執行)" --> PASS["✅ 語法完全合規，進入執行階段！"]
    B -- "是 (編譯期被攔截)" --> C["步驟 1: 看最底層錯誤訊息 (缺少 : / 括號 / invalid char)"]
    C --> D["步驟 2: 看行號與箭頭 ^ (確認當前行)"]
    D --> E{"當前行看起來完全正常?"}
    E -- "是" --> F["💡 抬頭看上一行！檢查上一行括號/引號是否未關閉！"]
    E -- "否" --> G["定位當前行符號 (冒號/等號/全形/運算子)"]
    F --> H["步驟 3: 單點修復，立刻重新執行編譯！"]
    G --> H
    H --> B
```

永遠遵循：**每次只修 1 處、循序推進、由上至下、絕不盲改**！


In [ ]:
# 13.2.9 程式碼演示：考場真實綜合多重語法地雷演練
# 模擬一段包含 5 種經典語法硬傷的學生解答代碼：
# 1. 漏冒號 2. 上一行漏右括號 3. 中文全形冒號 4. C語言 i++ 誤用 5. if 單等號

raw_exam_code = '''
def solve_problem(n)
    total = sum([1, 2, 3
    i = 0
    while i < n：
        if i = 3:
            total += 10
        i++
    return total
'''

print("=== 考場原始包含 5 重語法硬傷代碼 ===")
print(raw_exam_code)

def auto_exam_repair_agent(code):
    lines = code.split('\n')
    print("--- 啟動考場 1 分鐘 SOP 診斷修復流程 ---")
    
    # 修復 1: def 漏冒號
    lines[1] = "def solve_problem(n):"
    print("[修復 1] def solve_problem(n) -> 補上冒號 ':'")
    
    # 修復 2: 括號未閉合
    lines[2] = "    total = sum([1, 2, 3])"
    print("[修復 2] sum([1, 2, 3 -> 補上右中括號與右圓括號 '])'")
    
    # 修復 3: 中文冒號
    lines[4] = "    while i < n:"
    print("[修復 3] while i < n： -> 將全形冒號替換為半形 ':'")
    
    # 修復 4: if 單等號賦值
    lines[5] = "        if i == 3:"
    print("[修復 4] if i = 3: -> 將單等號改為雙等號 '=='")
    
    # 修復 5: C 語言 i++
    lines[7] = "        i += 1"
    print("[修復 5] i++ -> 改寫為標準 Python 'i += 1'")
    
    fixed_code = '\n'.join(lines)
    return fixed_code

clean_code = auto_exam_repair_agent(raw_exam_code)
print("\n=== 修復後代碼編譯驗證 ===")
compile(clean_code, "<repaired>", "exec")
print("🎉 恭喜！5 重語法地雷在 1 分鐘內全數清除，代碼完美通過編譯！")


### 13.2.9 語法重點回顧與核心觀念提煉

考場 1 分鐘語法除錯檢查清單（Checklist）：
1. **結構冒號**：`if`, `elif`, `else`, `for`, `while`, `def` 結尾全數有 `:`。
2. **括號配對**：每個 `(`, `[`, `{` 都有對應的 `)`, `]`, `}`，尤其是上一行結尾。
3. **字串引號**：單引號雙引號配對，跨多行文字使用三引號。
4. **縮排純潔**：全數使用 4 個半形空格，冒號下方若無邏輯補 `pass`。
5. **標點字元**：全數維持半形英文，無全形空白 `\u3000` 與中文標點。
6. **運算與賦值**：條件式雙等號 `==`，累加運算 `+= 1`（無 `i++`）。
7. **識別字規範**：變數名稱不與 Python 關鍵保留字衝突。


In [ ]:
# 13.2.9 學生實作練習：考場全方位語法醫生綜合驗收
# 任務說明：實作 fix_all_syntax_errors(broken_code) 函式
# 傳入一段具有多重典型語法缺陷的代碼，完成以下修復：
# 1. 將包含 'i++' 替換為 'i += 1'
# 2. 將中文全形冒號 '：' 替換為半形冒號 ':'
# 3. 將中文全形空白 '\u3000' 替換為半形空格 ' '
# 4. 將 if 語句中的單等號 " = " 替換為雙等號 " == "
# 回傳修復後的程式碼，確保能順利被 compile() 編譯！

def fix_all_syntax_errors(broken_code: str) -> str:
    # 執行全方位淨化
    cleaned = broken_code.replace("i++", "i += 1").replace("：", ":").replace("\u3000", " ")
    lines = cleaned.split('\n')
    for idx, l in enumerate(lines):
        if "if " in l and " = " in l and " == " not in l:
            lines[idx] = l.replace(" = ", " == ")
    return '\n'.join(lines)

# 測試用例
mock_broken = "if　val = 5：\n　　i++"
print("綜合修復結果:\n" + fix_all_syntax_errors(mock_broken))


In [ ]:
# 13.2.9 單元測試驗證
broken_test = "while　i < 10：\n　　if i = 2：\n　　　　pass\n　　i++"
fixed_test = fix_all_syntax_errors(broken_test)
assert "i++" not in fixed_test
assert "i += 1" in fixed_test
assert "：" not in fixed_test
assert "\u3000" not in fixed_test
assert "==" in fixed_test
compile(fixed_test, "<final_test>", "exec")
print("🎉 13.2.9 綜合實戰全數通過！恭喜你已修成橫掃全場 SyntaxError 的終極除錯宗師！")


## 13.2 總結與考場語法除錯終極全景對照表

在本單元中，我們將直譯器編譯檢查期的所有潛在障礙擴充梳理為 9 大核心子單元，將各類靜態語法與詞法錯誤一網打盡。送出代碼前，請隨時對照以下 7 大語法雷區速查圖譜：

| 語法雷區類別 | 典型報錯訊息 | 典型錯誤程式碼現場 | 考場 10 秒標準修復秘笈 |
| :--- | :--- | :--- | :--- |
| **結構漏冒號** | `SyntaxError: expected ':'` | `if x > 0` / `else` / `def f()` | 在 `if`, `elif`, `else`, `for`, `while`, `def` 結尾強制補上 `:` |
| **上一行漏括號** | `SyntaxError: invalid syntax`（游標指在下行） | 上行 `a = (1 + 2`，下行 `b = 3` | **抬頭看前一行！** 補齊遺失的右括號 `)` 或中括號 `]` |
| **括號引號錯配** | `closing parenthesis ']' does not match '('` | `(1, 2, 3]` / `'hello"` / `'It's'` | 左右括號同型對稱；含撇號文字外層強制使用雙引號 `"..."` |
| **縮排與 TabError** | `IndentationError` / `TabError` | 行首意外空格 / 冒號下方空行 / 混用 Tab | 頂格對齊；空區塊補 `pass`；編輯器設定 Tab 自動轉 4 個空格 |
| **全形空白/標點** | `SyntaxError: invalid character '　'` | 複製講義文字夾帶 `\u3000` 或中文冒號 `：` | 考場輸入法強制切為純英文半形；全文替換全形空格與標點 |
| **賦值與運算子** | `Maybe you meant '=='?` / `cannot assign` | `if x = 5:` / `10 = x` / `i++` | 條件式用雙等號 `==`；左側只放變數名；累加一律寫 `i += 1` |
| **關鍵字與 f-string** | `invalid syntax` (在 class=1 或 f"{x+}") | 保留字當變數名 / f-string 大括號算式未完 | 避開 35 個保留字；f-string 大括號內部必須為獨立完整合法表達式 |

### 🚀 下一步學習指引
當語法結構完全合規後，程式碼便能順利通過編譯期並啟動進入執行階段。然而，在程式跑動的中途，往往會因為數值不合常理（如陣列越界、字串轉整數失敗、查無此鍵、除以零）而突然暴斃！
在下一單元 **13-3《執行時期錯誤（Runtime Error, RE）常見排行榜與崩潰防禦》** 中，我們將直搗 APCS 考場五大執行崩潰殺手，建立密不透風的條件式防守心法！
